# 2026 COMP90042 Project

# Readme

**Environment**
- Google Colab, **Runtime → Change runtime type → T4 GPU** (required; the QLoRA
  verifier needs a GPU). CPU-only will not work.

**Data**
Place the provided dataset files in one of these locations:
- `./data/` , or `/content/data/` , or `/content/drive/MyDrive/comp90042/data/`

Required files: `train-claims.json`, `dev-claims.json`,
`test-claims-unlabelled.json`, and `evidence.json`.

**Caching (optional)**
If Google Drive is mounted, the notebook caches its built artefacts (BM25 index,
corpus embeddings, fine-tuned reranker, QLoRA adapter) to
`/content/drive/MyDrive/comp90042/cache/` and reuses them on later runs.

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [2]:
# =====================================================================
# 1. DATASET PROCESSING  +  EVIDENCE RETRIEVAL (first stage)
# ---------------------------------------------------------------------
# Task: for each climate claim, (a) retrieve evidence passages from a
# ~1.2M-passage corpus and (b) classify the claim into
# {SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED}.
#
# This cell: load data, preprocess, and build TWO complementary first-
# stage retrievers, then fuse them.
#   - DENSE bi-encoder (BGE): semantic / paraphrase match (claims rarely
#     share surface words with their evidence).
#   - BM25 (lexical): exact entity/number match the dense encoder blurs.
#   - Reciprocal Rank Fusion (RRF) merges both into one candidate pool.
# Rationale and ablations are in the report.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import importlib.util, subprocess, sys
def _ensure(pkgs):
    for mod, pip in pkgs:
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable,"-m","pip","install","-q",pip], check=True)
_ensure([("transformers","transformers==4.53.0"),("bm25s","bm25s"),
         ("Stemmer","PyStemmer"),("peft","peft"),("bitsandbytes","bitsandbytes"),
         ("accelerate","accelerate")])

import os, json, gc, random, time
import numpy as np, torch
SEED=13; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("device:", DEVICE, "| bf16:", USE_BF16)

# locate data + choose cache dir (Drive if available)
def _find_data_dir():
    for d in ["data","/content/data","/content/drive/MyDrive/comp90042/data"]:
        if os.path.exists(os.path.join(d,"evidence.json")): return d
    raise FileNotFoundError("Place evidence.json/train-claims.json/dev-claims.json in "
                            "./data or /content/drive/MyDrive/comp90042/data")
CACHE="cache"
try:
    from google.colab import drive; drive.mount("/content/drive")
    CACHE="/content/drive/MyDrive/comp90042/cache"; os.makedirs(CACHE,exist_ok=True)
    print("cache -> Drive:", CACHE)
except Exception:
    os.makedirs(CACHE,exist_ok=True); print("cache -> local:", CACHE)
DATA=_find_data_dir()
load_json=lambda p: json.load(open(p,encoding="utf-8"))
save_json=lambda o,p: json.dump(o,open(p,"w",encoding="utf-8"))

evidence    = load_json(os.path.join(DATA,"evidence.json"))
train_claims= load_json(os.path.join(DATA,"train-claims.json"))
dev_claims  = load_json(os.path.join(DATA,"dev-claims.json"))
_tp=os.path.join(DATA,"test-claims-unlabelled.json")
test_claims = load_json(_tp) if os.path.exists(_tp) else {}
print(f"evidence={len(evidence):,} train={len(train_claims)} dev={len(dev_claims)} test={len(test_claims)}")

LABELS=["SUPPORTS","REFUTES","NOT_ENOUGH_INFO","DISPUTED"]
LABEL2ID={l:i for i,l in enumerate(LABELS)}; ID2LABEL={i:l for l,i in LABEL2ID.items()}
EV_IDS=list(evidence.keys()); EV_TEXT=list(evidence.values())

# BM25 (lexical)
# Preprocessing here: lowercase + English stopwords + Porter stemming.
import bm25s, Stemmer
BM25_DIR=os.path.join(CACHE,"bm25s_index"); _stem=Stemmer.Stemmer("english")
if os.path.exists(BM25_DIR):
    bm25=bm25s.BM25.load(BM25_DIR,load_corpus=True); print("loaded BM25 index")
else:
    print("building BM25 index ...")
    toks=bm25s.tokenize(EV_TEXT,stopwords="en",stemmer=_stem,show_progress=True)
    bm25=bm25s.BM25(corpus=EV_IDS); bm25.index(toks); bm25.save(BM25_DIR)
def bm25_search(claims, top_k=100):
    out={}
    for cid,c in claims.items():
        q=bm25s.tokenize(c["claim_text"],stopwords="en",stemmer=_stem,show_progress=False)
        d,_=bm25.retrieve(q,k=top_k,show_progress=False)
        out[cid]=[(d[0,r]["text"] if isinstance(d[0,r],dict) else d[0,r]) for r in range(d.shape[1])]
    return out

# Dense (BGE bi-encoder)
# Preprocessing: raw text; claims get BGE's query instruction prefix.
from transformers import AutoTokenizer, AutoModel
BGE="BAAI/bge-small-en-v1.5"
QPFX="Represent this sentence for searching relevant passages: "
EMB=os.path.join(CACHE,"corpus_emb_bge_small.npy")
_bt=AutoTokenizer.from_pretrained(BGE); _bm=AutoModel.from_pretrained(BGE).to(DEVICE).eval()
@torch.no_grad()
def bge_encode(texts, prefix="", bs=256, maxlen=160):
    o=[]
    for i in range(0,len(texts),bs):
        b=_bt([prefix+t for t in texts[i:i+bs]],padding=True,truncation=True,
              max_length=maxlen,return_tensors="pt").to(_bm.device)
        o.append(torch.nn.functional.normalize(_bm(**b).last_hidden_state[:,0],dim=1).cpu().numpy().astype(np.float32))
    return np.concatenate(o)
if os.path.exists(EMB):
    corpus_emb=np.load(EMB,mmap_mode="r"); print("loaded corpus emb", corpus_emb.shape)
else:
    print("encoding corpus with bge-small (~15-20 min on T4)...")
    n,dim=len(EV_TEXT),_bm.config.hidden_size
    corpus_emb=np.lib.format.open_memmap(EMB,mode="w+",dtype=np.float32,shape=(n,dim))
    t0=time.time()
    for s in range(0,n,50000):
        e=min(s+50000,n); corpus_emb[s:e]=bge_encode(EV_TEXT[s:e]); corpus_emb.flush()
        print(f"  {e:,}/{n:,} ({time.time()-t0:.0f}s)")
def dense_search(claims, top_k=100):
    cids=list(claims.keys()); q=bge_encode([claims[c]["claim_text"] for c in cids],prefix=QPFX)
    out={}
    for ci,cid in enumerate(cids):
        sims=corpus_emb@q[ci]; idx=np.argpartition(-sims,top_k)[:top_k]; idx=idx[np.argsort(-sims[idx])]
        out[cid]=[EV_IDS[j] for j in idx]
    return out

# Fuse (RRF)
def rrf(*ranked,k=60,top_k=100):
    out={}
    for cid in ranked[0]:
        agg={}
        for rl in ranked:
            for rank,eid in enumerate(rl.get(cid,[])): agg[eid]=agg.get(eid,0.0)+1.0/(k+rank+1)
        out[cid]=[e for e,_ in sorted(agg.items(),key=lambda x:-x[1])[:top_k]]
    return out
def build_pool(claims,top_k=100): return rrf(dense_search(claims,top_k),bm25_search(claims,top_k),top_k=top_k)
def cached_pool(name,claims):
    p=os.path.join(CACHE,f"{name}_rrf100.json")
    if os.path.exists(p): return load_json(p)
    pool=build_pool(claims); save_json(pool,p); return pool
train_pool=cached_pool("train",train_claims); dev_pool=cached_pool("dev",dev_claims)
def recall_at(pool,claims,k):
    return float(np.mean([len(set(c["evidences"])&set(pool[cid][:k]))/len(c["evidences"])
                          for cid,c in claims.items() if c["evidences"]]))
print(f"dev pool recall@100 = {recall_at(dev_pool,dev_claims,100):.3f}")


device: cuda | bf16: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache -> Drive: /content/drive/MyDrive/comp90042/cache
evidence=1,208,827 train=1228 dev=154 test=153
loaded BM25 index


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded corpus emb (1208827, 384)
dev pool recall@100 = 0.614


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [3]:
# =====================================================================
# 2. MODEL IMPLEMENTATION
#   (a) Reranker: cross-encoder fine-tuned with a LISTWISE objective to
#       order the RRF pool.
#   (b) Classifier: QLoRA-tuned Qwen2.5-1.5B used as a verifier that reads
#       the claim + top retrieved evidence and emits a label, scored by
#       comparing the next-token logits of A/B/C/D.
# All training is on RETRIEVED evidence so train matches test.
from transformers import AutoModelForSequenceClassification

# (a) LISTWISE reranker fine-tune
# Per claim: score gold-in-pool (positives) vs sampled hard negatives from
# the SAME pool; softmax over the group; loss = -log sum P(positives).
# This directly optimises "rank gold above its competitors" (what F@k rewards).
RR_BASE="cross-encoder/ms-marco-MiniLM-L-6-v2"
RR_DIR=os.path.join(CACHE,"reranker_ft"); GROUP_NEG=19; RR_MAXLEN=256
def _rr_groups(claims,pool):
    g=[]
    for cid,c in claims.items():
        gold=set(c["evidences"]); cand=pool[cid]
        pos=[e for e in cand if e in gold]
        if not pos: continue
        negs=[e for e in cand if e not in gold]; random.shuffle(negs); negs=negs[:GROUP_NEG]
        g.append((c["claim_text"],[(e,1.0) for e in pos]+[(e,0.0) for e in negs]))
    return g
def train_reranker(epochs=3, lr=2e-5):
    if os.path.exists(os.path.join(RR_DIR,"model.safetensors")):
        print("loaded cached fine-tuned reranker"); return
    tok=AutoTokenizer.from_pretrained(RR_BASE)
    model=AutoModelForSequenceClassification.from_pretrained(RR_BASE,num_labels=1).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    amp=torch.bfloat16 if USE_BF16 else torch.float16
    scaler=torch.amp.GradScaler("cuda",enabled=not USE_BF16)
    groups=_rr_groups(train_claims,train_pool); print(f"reranker groups={len(groups)}")
    model.train()
    for ep in range(epochs):
        random.shuffle(groups); run=0.0
        for gi,(claim,items) in enumerate(groups):
            enc=tok([claim]*len(items),[evidence[e] for e,_ in items],truncation=True,
                    max_length=RR_MAXLEN,padding=True,return_tensors="pt").to(DEVICE)
            lab=torch.tensor([y for _,y in items],device=DEVICE)
            opt.zero_grad()
            with torch.autocast("cuda",dtype=amp):
                logits=model(**enc).logits.squeeze(-1).float()
                logp=torch.log_softmax(logits,0)
                loss=-torch.logsumexp(logp[lab>0.5],0)
            if USE_BF16: loss.backward(); opt.step()
            else: scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run+=loss.item()
        print(f"  reranker epoch {ep} loss={run/len(groups):.4f}")
    os.makedirs(RR_DIR,exist_ok=True); model.save_pretrained(RR_DIR); tok.save_pretrained(RR_DIR)
    del model; gc.collect(); torch.cuda.empty_cache()
train_reranker()

# apply reranker -> ranked top-k per claim (cached)
_rtok=AutoTokenizer.from_pretrained(RR_DIR)
_rmodel=AutoModelForSequenceClassification.from_pretrained(RR_DIR).to(DEVICE).eval()
@torch.no_grad()
def rerank(claims,pool,top_k=10):
    out={}
    for cid,c in claims.items():
        cand=pool[cid]; sc=[]
        for i in range(0,len(cand),64):
            ch=cand[i:i+64]
            enc=_rtok([c["claim_text"]]*len(ch),[evidence[e] for e in ch],padding=True,
                      truncation=True,max_length=RR_MAXLEN,return_tensors="pt").to(_rmodel.device)
            lg=_rmodel(**enc).logits
            sc.extend((lg[:,0] if lg.shape[-1]==1 else lg.squeeze(-1)).float().cpu().tolist())
        out[cid]=[e for e,_ in sorted(zip(cand,sc),key=lambda x:-x[1])[:top_k]]
    return out
def cached_rank(name,claims):
    p=os.path.join(CACHE,f"{name}_ranked.json")
    if os.path.exists(p): return load_json(p)
    r=rerank(claims,train_pool if name=="train" else dev_pool); save_json(r,p); return r
train_ranked=cached_rank("train",train_claims); dev_ranked=cached_rank("dev",dev_claims)
print(f"dev reranked recall@3 = {recall_at(dev_ranked,dev_claims,3):.3f}")

# (b) QLoRA Qwen2.5-1.5B verifier/classifier
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
QWEN="Qwen/Qwen2.5-1.5B-Instruct"
LET=["A","B","C","D"]
LET2LAB=dict(zip(LET,LABELS)); LAB2LET={v:k for k,v in LET2LAB.items()}
QMAXLEN=384; READ_N=3; EMIT_N=3; LORA_DIR=os.path.join(CACHE,"qwen_lora")
def _prompt(claim,eids):
    ev="\n".join(f"- {evidence[e][:200]}" for e in eids if e in evidence) or "- (none)"
    return ("You are a climate-claim fact-checker. Using ONLY the evidence, choose the label.\n"
            f"Claim: {claim}\nEvidence:\n{ev}\n\n"
            "A) SUPPORTS  B) REFUTES  C) NOT_ENOUGH_INFO  D) DISPUTED\n"
            "Answer with a single letter (A/B/C/D).")
_qtok=AutoTokenizer.from_pretrained(QWEN,trust_remote_code=True)
if _qtok.pad_token_id is None: _qtok.pad_token=_qtok.eos_token
def _load_qwen_4bit():
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16)
    return AutoModelForCausalLM.from_pretrained(QWEN,quantization_config=bnb,
        device_map="auto",trust_remote_code=True)
def train_qwen(epochs=3, lr=2e-4):
    if os.path.exists(os.path.join(LORA_DIR,"adapter_config.json")):
        print("loaded cached QLoRA adapter"); return
    base=_load_qwen_4bit(); base.config.use_cache=False
    base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model=get_peft_model(base,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",
        task_type="CAUSAL_LM",target_modules=["q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"]))
    rows=[]
    for cid,c in train_claims.items():
        rows.append((c["claim_text"],train_ranked[cid][:READ_N],c["claim_label"]))
        if c["evidences"]: rows.append((c["claim_text"],c["evidences"],c["claim_label"]))
    try:
        import bitsandbytes as _bnb
        opt=_bnb.optim.PagedAdamW8bit([p for p in model.parameters() if p.requires_grad],lr=lr)
    except Exception:
        opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr)
    amp=torch.bfloat16 if USE_BF16 else torch.float16
    scaler=torch.amp.GradScaler("cuda",enabled=not USE_BF16)
    model.train(); GA=16; CLIP=1.0
    for ep in range(epochs):
        random.shuffle(rows); opt.zero_grad(); run=0.0; pending=False
        for i,(claim,ids,lab) in enumerate(rows):
            msgs=[{"role":"user","content":_prompt(claim,ids)}]
            pre=_qtok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
            pid=_qtok(pre,add_special_tokens=False).input_ids
            aid=_qtok(LAB2LET[lab],add_special_tokens=False).input_ids
            pid=pid[:QMAXLEN-len(aid)]
            ids_t=torch.tensor([pid+aid],device=model.device)
            lbl=torch.tensor([[-100]*len(pid)+aid],device=model.device)
            with torch.autocast("cuda",dtype=amp):
                out=model(input_ids=ids_t,labels=lbl)
            if not torch.isfinite(out.loss):
                continue
            scaler.scale(out.loss/GA).backward(); run+=out.loss.item(); pending=True
            if (i+1)%GA==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(); pending=False
        if pending:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        print(f"  qwen epoch {ep} loss={run/len(rows):.4f}")
    model.save_pretrained(LORA_DIR); del model,base; gc.collect(); torch.cuda.empty_cache()
try:
    _rmodel.to('cpu'); _bm.to('cpu')
except Exception:
    pass
import gc as _gc; _gc.collect(); torch.cuda.empty_cache()
train_qwen()

_qwen=PeftModel.from_pretrained(_load_qwen_4bit(),LORA_DIR).eval()
_LET_IDS=[_qtok(l,add_special_tokens=False).input_ids[0] for l in LET]
@torch.no_grad()
def qwen_classify(claims,ranked,read_n=READ_N):
    pred={}
    for cid,c in claims.items():
        msgs=[{"role":"user","content":_prompt(c["claim_text"],ranked[cid][:read_n])}]
        text=_qtok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
        inp=_qtok(text,return_tensors="pt").to(_qwen.device)
        lg=_qwen(**inp).logits[0,-1]
        pred[cid]=LET2LAB[LET[int(torch.tensor([lg[t] for t in _LET_IDS]).argmax())]]
    return pred
print("model components ready.")


loaded cached fine-tuned reranker


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

dev reranked recall@3 = 0.278


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model components ready.


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [4]:
# =====================================================================
# 3. TESTING AND EVALUATION
# Official metric (eval.py): per-claim evidence F-score (F), claim
# accuracy (A), and their harmonic mean (H). Produces dev scores, a
# per-class breakdown, and the test-output.json for the leaderboard.
def score(predictions, groundtruth):
    fs, accs = [], []
    for cid, claim in groundtruth.items():
        p = predictions.get(cid)
        if not p: continue
        accs.append(1.0 if p["claim_label"]==claim["claim_label"] else 0.0)
        f=0.0; pe=p.get("evidences",[])
        if pe:
            hit=len(set(claim["evidences"])&set(pe))
            if hit:
                r=hit/len(claim["evidences"]); pr=hit/len(set(pe)); f=2*pr*r/(pr+r)
        fs.append(f)
    F=float(np.mean(fs)); A=float(np.mean(accs)); H=0.0 if F+A==0 else 2*F*A/(F+A)
    return F, A, H

def predict(claims, ranked):
    labels = qwen_classify(claims, ranked)
    return {cid: {"claim_label": labels[cid], "evidences": ranked[cid][:EMIT_N]}
            for cid in claims}

dev_pred = predict(dev_claims, dev_ranked)
F, A, H = score(dev_pred, dev_claims)
print(f"DEV:  F(evidence)={F:.4f}  A(claim acc)={A:.4f}  H(harmonic mean)={H:.4f}")

from collections import Counter
tot, cor = Counter(), Counter()
for cid, c in dev_claims.items():
    tot[c["claim_label"]] += 1
    cor[c["claim_label"]] += int(dev_pred[cid]["claim_label"]==c["claim_label"])
print("per-class accuracy:")
for l in LABELS:
    if tot[l]: print(f"  {l:16s} {cor[l]:3d}/{tot[l]:3d} = {cor[l]/tot[l]:.2f}")

DEV:  F(evidence)=0.2457  A(claim acc)=0.6234  H(harmonic mean)=0.3525
per-class accuracy:
  SUPPORTS          54/ 68 = 0.79
  REFUTES           12/ 27 = 0.44
  NOT_ENOUGH_INFO   26/ 41 = 0.63
  DISPUTED           4/ 18 = 0.22


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*